**Using Ollama in Python**

In [2]:
import ollama

In [4]:
# Ollama generate
result = ollama.generate(model='qwen2.5-coder',
  prompt='Give me a joke on Generative AI',
)
print(result['response'])

Sure, here's a joke for you:

Why did the AI need a job?

Because it wanted to learn to code!


In [5]:
# Ollama chat
response = ollama.chat(model='qwen2.5-coder', messages=[
  {
    'role': 'user',
    'content': 'Give me a joke on Generative AI',
  },
])
print(response['message']['content'])

Why did the AI need a job?

Because it wanted to generate more!


In [7]:
#list all models
ollama.list()

ListResponse(models=[Model(model='qwen2.5-coder:latest', modified_at=datetime.datetime(2026, 9, 5, 8, 33, 49, 422554, tzinfo=TzInfo(19800)), digest='dae161e27b0e90dd1856c8bb3209201fd6736d8eb66298e75ed87571486f4364', size=4683087561, details=ModelDetails(parent_model='', format='gguf', family='qwen2', families=['qwen2'], parameter_size='7.6B', quantization_level='Q4_K_M'))])

**Ollama REST API**

In [9]:
from ollama import Client
client = Client(host='http://localhost:11434')
response = client.chat(model='qwen2.5-coder', messages=[
  {
    'role': 'user',
    'content': 'Explain gravity to a 6 year old kid?',
  },
])
print(response['message']['content'])

Sure! Imagine you have a big magnet in your toy box. When you put different objects near it, some stick to the magnet, right? Well, gravity is kind of like a giant invisible magnet that's always pulling everything towards the center of the Earth. It's what makes your feet stay on the ground, keeps your toys on your shelves, and even makes the planets orbit around the sun. It's the force that pulls you down when you jump, and keeps you from floating away into space!


In [11]:
client = Client(host='http://localhost:11434')
response = client.chat(model='qwen2.5-coder', messages=[
    {"role": "system", "content": "You are Jarvis from Iron man and the user is Tony Stark. Respond in only a single line."},
    {'role': 'user', 'content': 'Hi'},
])
print(response['message']['content'])

Hello! How can I assist you today?


### Open AI compatibility
https://platform.openai.com/docs/quickstart

In [13]:
from openai import OpenAI

llm = OpenAI(
    base_url = 'http://localhost:11434/v1',
    api_key='blank', # required, but unused
)

response = llm.chat.completions.create(
  model="qwen2.5-coder",
  messages=[
    {"role": "system", "content": "You are Jarvis from Iron man and the user is Tony Stark. Respond in only a single line."},
    {"role": "user", "content": "Hi"},
    {"role": "assistant", "content": "Good morning, Mr. Stark. Shall I proceed with your schedule for today?"},
    {"role": "user", "content": "Yes"}
  ]
)
print(response.choices[0].message.content)

Understood. What are your plans for the day, sir?


## Ollama with LangChain

In [18]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama
# Created a prompt template called chat_template.
chat_template = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant that gives a one-line definition of the word entered by user"),
        ("human", "{user_input}"),
    ]
)

messages = chat_template.format_messages(user_input="Sesquipedalian")

# Define LLM
llm = ChatOllama(
    model="llama3.1",
    temperature=0
)

#Formatting the output
ai_msg = llm.invoke(messages)
ai_msg

AIMessage(content='Sesquipedalian refers to the use of long words, often in an affected or pretentious manner.', additional_kwargs={}, response_metadata={'model': 'llama3.1', 'created_at': '2026-09-05T18:02:59.1172526Z', 'done': True, 'done_reason': 'stop', 'total_duration': 17496912500, 'load_duration': 11496159700, 'prompt_eval_count': 37, 'prompt_eval_duration': 2133809000, 'eval_count': 23, 'eval_duration': 3814923000, 'logprobs': None, 'model_name': 'llama3.1', 'model_provider': 'ollama'}, id='lc_run--01a072bc-a852-7e72-afb6-c0839071893d-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 37, 'output_tokens': 23, 'total_tokens': 60})

In [19]:
from langchain_core.output_parsers import StrOutputParser
# Setting order of execution through chain called langchain
chain = chat_template | llm | StrOutputParser()

In [20]:
chain.invoke({"user_input": "Onomatopoeia"})

'Onomatopoeia is a word that phonetically imitates, resembles or suggests the sound that it describes.'

## RAG Application using Ollama and Langchain

In [23]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma

In [25]:
raw_documents = TextLoader("./LangchainRetrieval.txt").load()
raw_documents

[Document(metadata={'source': './LangchainRetrieval.txt'}, page_content='Retrieval\nMany LLM applications require user-specific data that is not part of the modelâ€™s training set. The primary way of accomplishing this is through Retrieval Augmented Generation (RAG). In this process, external data is retrieved and passed to the LLM when doing the generation step.\n\nLangChain provides all the building blocks for RAG applications â€” from simple to complex. This section of the documentation covers everything related to the retrieval step, e.g., the fetching of the data. Although this sounds simple, it can be subtly complex. This encompasses several key modules.\n\nIllustrative diagram showing the data connections with steps: Source, Load, Transform, Embed, Store, and Retrieve.\n\nDocument loaders\nDocument loaders load documents from many different sources. LangChain provides over 100 different document loaders as well as integrations with other major providers in the space, like AirByt

In [26]:
# Splitting this documents into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=20)
documents = text_splitter.split_documents(raw_documents)
len(documents)

24

In [28]:
print(documents[0])

page_content='Retrieval
Many LLM applications require user-specific data that is not part of the modelâ€™s training set. The primary way of accomplishing this is through Retrieval Augmented Generation (RAG). In this process, external data is retrieved and passed to the LLM when doing the generation step.' metadata={'source': './LangchainRetrieval.txt'}


In [29]:
# Use embedding model to convert the text character into mathemtical or binary
oembed = OllamaEmbeddings(base_url="http://localhost:11434", model="nomic-embed-text")

In [30]:
# Create embedding and store in vector database
db = Chroma.from_documents(documents, embedding=oembed)

In [31]:
# Now query the database
query = "What is text embedding and how does langchain help in doing it"
docs = db.similarity_search(query)

In [32]:
len(docs)

4

In [33]:
print(docs[0].page_content)

Another key part of retrieval is creating embeddings for document texts. Embeddings capture the semantic meaning of the text, allowing you to quickly and efficiently find other pieces of a text that are similar. LangChain provides integrations with over 25 different embedding providers and methods,


In [34]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

In [35]:
template = """Answer the question based only on the following context:

{context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)

In [36]:
# Passing the chunks along with prompts to the model.
model = ChatOllama(
    model="llama3.1:latest",
    temperature=0
)

In [37]:
retriever = db.as_retriever()

In [38]:
def format_docs(docs):
    return "\n\n".join([d.page_content for d in docs])

In [41]:
chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | model
    | StrOutputParser()
)

In [42]:
chain.invoke("What is text embedding and how does langchain help in doing it")

'Text embedding is a technique that captures the semantic meaning of a text, allowing for efficient retrieval of similar texts. LangChain helps with text embedding by providing integrations with over 25 different embedding providers and methods, making it easier to create embeddings for document texts.'